# 2. Data Cleaning (Limpieza de Datos)

## Objetivo
Aplicar transformaciones y correcciones para garantizar la calidad de los datos:
- Eliminar/imputar valores nulos
- Remover duplicados
- Tratar outliers
- Normalizar y validar
- Estandarizar formatos

## Entrada
Datos crudos de `data/00_raw/traffic_data.csv`

## Salida
Dataset limpio en `data/02_clean/traffic_data_clean.csv`

## 2.1 Importar librerías

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

print('✅ Librerías importadas')

## 2.2 Cargar datos crudos

In [ ]:
# Cargar datos
DATA_RAW_PATH = '../../data/00_raw/traffic_data.csv'
df = pd.read_csv(DATA_RAW_PATH)

print(f'Datos cargados: {len(df)} registros × {len(df.columns)} columnas')
print(f'Memoria: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')

# Guardar conteos iniciales para comparar después
registros_iniciales = len(df)

## 2.3 Paso 1: Conversión de tipos de datos

In [ ]:
print('CONVERSIÓN DE TIPOS DE DATOS:\n')

# Convertir timestamp a datetime
if 'timestamp' in df.columns:
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        print('✅ timestamp convertido a datetime')
    except Exception as e:
        print(f'⚠️  Error al convertir timestamp: {e}')

# Asegurar que columnas numéricas sean numéricas
columnas_numericas = ['velocidad', 'densidad', 'detenciones']
for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'✅ {col} convertido a numérico')

# Coordenadas geográficas
for col in ['latitud', 'longitud']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'✅ {col} convertido a numérico')

print(f'\nTipos finales:\n{df.dtypes}')

## 2.4 Paso 2: Manejo de valores nulos

In [ ]:
print('MANEJO DE VALORES NULOS:\n')

nulos_antes = df.isnull().sum().sum()
print(f'Valores nulos antes: {nulos_antes}')

# Estrategia 1: Columnas críticas (sin nulos)
columnas_criticas = ['timestamp', 'avenida']
for col in columnas_criticas:
    if col in df.columns:
        nulos_col = df[col].isnull().sum()
        if nulos_col > 0:
            print(f'⚠️  Eliminando {nulos_col} filas con {col} nulo')
            df = df[df[col].notna()]

# Estrategia 2: Variables numéricas (imputar con media por avenida)
for col in ['velocidad', 'densidad', 'detenciones']:
    if col in df.columns and df[col].isnull().sum() > 0:
        print(f'⚠️  Imputando {col} con media por avenida')
        df[col] = df.groupby('avenida')[col].transform(
            lambda x: x.fillna(x.mean())
        )
        # Si aún quedan nulos, usar media global
        df[col].fillna(df[col].mean(), inplace=True)

# Estrategia 3: Descripción (valores por defecto)
if 'descripcion' in df.columns:
    df['descripcion'].fillna('Sin descripción', inplace=True)

nulos_despues = df.isnull().sum().sum()
print(f'\nValores nulos después: {nulos_despues}')
print(f'✅ Reducción: {nulos_antes - nulos_despues} nulos eliminados')

## 2.5 Paso 3: Remover duplicados

In [ ]:
print('REMOCIÓN DE DUPLICADOS:\n')

duplicados_antes = df.duplicated().sum()
print(f'Duplicados totales: {duplicados_antes}')

# Remover duplicados exactos
if duplicados_antes > 0:
    df = df.drop_duplicates()
    print(f'✅ Duplicados exactos removidos')

# Remover duplicados por timestamp + avenida (mismo evento reportado)
duplicados_evento = df.duplicated(subset=['timestamp', 'avenida']).sum()
if duplicados_evento > 0:
    print(f'\n⚠️  Duplicados por timestamp+avenida: {duplicados_evento}')
    print('Estrategia: Mantener el primer registro (más probable que sea correcto)')
    df = df.drop_duplicates(subset=['timestamp', 'avenida'], keep='first')
    print(f'✅ {duplicados_evento} registros duplicados removidos')

duplicados_despues = df.duplicated().sum()
print(f'\nDuplicados después: {duplicados_despues}')

## 2.6 Paso 4: Validación y corrección de rangos

In [ ]:
print('VALIDACIÓN Y CORRECCIÓN DE RANGOS:\n')

# Velocidad: debe estar entre 0 y 120 km/h
if 'velocidad' in df.columns:
    velocidad_invalida = ((df['velocidad'] < 0) | (df['velocidad'] > 120)).sum()
    if velocidad_invalida > 0:
        print(f'⚠️  {velocidad_invalida} registros con velocidad inválida')
        # Limitar valores al rango válido
        df['velocidad'] = df['velocidad'].clip(0, 120)
        print(f'✅ Velocidades limitadas al rango [0, 120]')

# Densidad: debe estar entre 1 y 3
if 'densidad' in df.columns:
    densidad_invalida = ((df['densidad'] < 1) | (df['densidad'] > 3)).sum()
    if densidad_invalida > 0:
        print(f'⚠️  {densidad_invalida} registros con densidad inválida')
        # Redondear al rango válido
        df['densidad'] = df['densidad'].round().clip(1, 3).astype(int)
        print(f'✅ Densidades ajustadas al rango [1, 3]')

# Detenciones: debe estar entre 0 y 30
if 'detenciones' in df.columns:
    detenciones_invalida = ((df['detenciones'] < 0) | (df['detenciones'] > 30)).sum()
    if detenciones_invalida > 0:
        print(f'⚠️  {detenciones_invalida} registros con detenciones inválidas')
        df['detenciones'] = df['detenciones'].clip(0, 30).astype(int)
        print(f'✅ Detenciones limitadas al rango [0, 30]')

# Coordenadas geográficas
if 'latitud' in df.columns:
    lat_invalida = ((df['latitud'] < -90) | (df['latitud'] > 90)).sum()
    if lat_invalida > 0:
        print(f'⚠️  {lat_invalida} registros con latitud inválida')
        df = df[(df['latitud'] >= -90) & (df['latitud'] <= 90)]

if 'longitud' in df.columns:
    lon_invalida = ((df['longitud'] < -180) | (df['longitud'] > 180)).sum()
    if lon_invalida > 0:
        print(f'⚠️  {lon_invalida} registros con longitud inválida')
        df = df[(df['longitud'] >= -180) & (df['longitud'] <= 180)]

print('\n✅ Rangos validados y corregidos')

## 2.7 Paso 5: Tratar outliers

In [ ]:
print('TRATAMIENTO DE OUTLIERS (Método IQR):\n')

def tratar_outliers_iqr(data, columna, multiplicador=1.5):
    """Limita outliers usando el método IQR"""
    Q1 = data[columna].quantile(0.25)
    Q3 = data[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - multiplicador * IQR
    limite_sup = Q3 + multiplicador * IQR
    
    outliers = ((data[columna] < limite_inf) | (data[columna] > limite_sup)).sum()
    
    if outliers > 0:
        data[columna] = data[columna].clip(limite_inf, limite_sup)
    
    return outliers, limite_inf, limite_sup

# Aplicar a variables numéricas clave
for col in ['velocidad', 'detenciones']:
    if col in df.columns:
        outliers, limite_inf, limite_sup = tratar_outliers_iqr(df, col)
        print(f'{col}: {outliers} outliers tratados')
        print(f'  Límites: [{limite_inf:.2f}, {limite_sup:.2f}]\n')

print('✅ Outliers limitados mediante clipping')

## 2.8 Paso 6: Limpieza de columnas categóricas

In [ ]:
print('LIMPIEZA DE COLUMNAS CATEGÓRICAS:\n')

# Avenida: eliminar espacios y estandarizar
if 'avenida' in df.columns:
    df['avenida'] = df['avenida'].str.strip().str.title()
    print(f'Avenidas únicas: {df["avenida"].nunique()}')
    print(f'  {list(df["avenida"].unique())}')

# Descripción: eliminar espacios extra
if 'descripcion' in df.columns:
    df['descripcion'] = df['descripcion'].str.strip()
    print(f'\n✅ Descripciones limpiadas')

print('\n✅ Columnas categóricas limpias')

## 2.9 Paso 7: Ordenar y reorganizar datos

In [ ]:
print('REORGANIZACIÓN DE DATOS:\n')

# Ordenar por timestamp y avenida
if 'timestamp' in df.columns:
    df = df.sort_values(['timestamp', 'avenida']).reset_index(drop=True)
    print('✅ Datos ordenados por timestamp y avenida')

# Asegurar orden de columnas consistente
orden_columnas = ['timestamp', 'avenida', 'latitud', 'longitud', 
                  'velocidad', 'densidad', 'detenciones', 'horario', 'descripcion']
columnas_presentes = [col for col in orden_columnas if col in df.columns]
df = df[columnas_presentes]

print(f'✅ Columnas reorganizadas: {columnas_presentes}')

## 2.10 Resumen de cambios

In [ ]:
print('='*80)
print('RESUMEN DE LIMPIEZA')
print('='*80)

registros_finales = len(df)
registros_eliminados = registros_iniciales - registros_finales
porcentaje_retenido = (registros_finales / registros_iniciales * 100)

print(f'\n📊 ESTADÍSTICAS:')
print(f'  Registros iniciales: {registros_iniciales}')
print(f'  Registros finales: {registros_finales}')
print(f'  Registros eliminados: {registros_eliminados}')
print(f'  Porcentaje retenido: {porcentaje_retenido:.2f}%')

print(f'\n📝 DATOS LIMPIOS:')
print(f'  Dimensiones: {registros_finales} × {len(df.columns)}')
print(f'  Columnas: {list(df.columns)}')
print(f'  Valores nulos: {df.isnull().sum().sum()}')
print(f'  Duplicados: {df.duplicated().sum()}')

print(f'\n📅 RANGO TEMPORAL:')
if 'timestamp' in df.columns:
    print(f'  Inicio: {df["timestamp"].min()}')
    print(f'  Fin: {df["timestamp"].max()}')

## 2.11 Guardar datos limpios

In [ ]:
# Crear directorio si no existe
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardar datos limpios
OUTPUT_PATH = '../../data/02_clean/traffic_data_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Datos limpios guardados en: {OUTPUT_PATH}')
print(f'\nTamaño del archivo: {os.path.getsize(OUTPUT_PATH) / 1024:.2f} KB')

# Guardar también en formato pickle para análisis posteriores
pickle_path = '../../data/02_clean/traffic_data_clean.pkl'
df.to_pickle(pickle_path)
print(f'✅ Backup en formato pickle: {pickle_path}')

## 2.12 Próximos pasos

In [ ]:
print('\n' + '='*80)
print('LIMPIEZA COMPLETADA ✅')
print('='*80)
print('\n📋 Próximos pasos:')
print('  1. Ejecutar: Data_Validation.ipynb (Verificar integridad post-limpieza)')
print('  2. Ejecutar: Exploratory_Data_Analysis.ipynb (Descubrir patrones)')
print('  3. Empezar feature engineering para construir la métrica TSI')
print('\n✅ El dataset está listo para análisis exploratorio y modelado')